In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_train_path = r'../../IEEE-CIS-Fraud-Detection/Data/01_Raw_data/home-credit-default-risk/application_train.csv'
df = pd.read_csv(df_train_path)

In [ ]:
df_cols = df.columns
for i, col in enumerate(df_cols):
    missing = str(df[col].isnull().sum()).rjust(10)  # căn phải trong khung rộng 10 ký tự
    print(f"{i:<3} {col:<30} {str('|').rjust(5)} {missing}")

In [ ]:
# xóa cột id vì nó không có ý nghĩa trong việc huấn luyện model
df = df.drop(columns = ['SK_ID_CURR'])

In [ ]:
def fillna_with_mean(col_name):
    df[col_name] =df[col_name].fillna(df[col_name].mean())
def fillna_with_median(col_name):
    df[col_name] =df[col_name].fillna(df[col_name].median())
def fillna_with_mode(col_name):
    df[col_name] = df[col_name].fillna(df[col_name].mode()[0])


In [ ]:
fill_mean_cols = ['AMT_ANNUITY','AMT_GOODS_PRICE']
fill_median_cols = []
fill_mode_cols = [
    'OCCUPATION_TYPE','CNT_FAM_MEMBERS','OBS_30_CNT_SOCIAL_CIRCLE',
    "DEF_30_CNT_SOCIAL_CIRCLE", "OBS_60_CNT_SOCIAL_CIRCLE", "DEF_30_CNT_SOCIAL_CIRCLE",'DEF_60_CNT_SOCIAL_CIRCLE',
    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK', 
    'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR','YEARS_BEGINEXPLUATATION_AVG']
delete_values_nah = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','DAYS_LAST_PHONE_CHANGE']  # chỉ xóa các dữ liệu phổ biến
delete_cols = [
    'NAME_TYPE_SUITE','WEEKDAY_APPR_PROCESS_START','HOUR_APPR_PROCESS_START',
    'APARTMENTS_AVG','BASEMENTAREA_AVG','YEARS_BUILD_AVG',
    'COMMONAREA_AVG','ELEVATORS_AVG','ENTRANCES_AVG','FLOORSMAX_AVG','FLOORSMIN_AVG',
    'LANDAREA_AVG','LIVINGAPARTMENTS_AVG','LIVINGAREA_AVG','NONLIVINGAPARTMENTS_AVG',
    'NONLIVINGAREA_AVG','APARTMENTS_MODE','BASEMENTAREA_MODE','YEARS_BEGINEXPLUATATION_MODE',
    'YEARS_BUILD_MODE','COMMONAREA_MODE','ELEVATORS_MODE','ENTRANCES_MODE','FLOORSMAX_MODE',
    'FLOORSMIN_MODE','LANDAREA_MODE','LIVINGAPARTMENTS_MODE','LIVINGAREA_MODE',
    'NONLIVINGAPARTMENTS_MODE','NONLIVINGAREA_MODE','APARTMENTS_MEDI','BASEMENTAREA_MEDI',
    'YEARS_BEGINEXPLUATATION_MEDI','YEARS_BUILD_MEDI','COMMONAREA_MEDI','ELEVATORS_MEDI',
    'ENTRANCES_MEDI','FLOORSMAX_MEDI','FLOORSMIN_MEDI','LANDAREA_MEDI','LIVINGAPARTMENTS_MEDI',
    'LIVINGAREA_MEDI','NONLIVINGAPARTMENTS_MEDI','NONLIVINGAREA_MEDI','FONDKAPREMONT_MODE',
    'HOUSETYPE_MODE','TOTALAREA_MODE','WALLSMATERIAL_MODE','EMERGENCYSTATE_MODE'
]


In [ ]:
df['OWN_CAR_AGE'] = df['OWN_CAR_AGE'].fillna(1)

In [ ]:
for col in fill_mean_cols:
    fillna_with_mean(col)

print(df[fill_mean_cols].head())
print(df[fill_mean_cols].isnull().sum())

In [ ]:
for col in fill_mode_cols:
    fillna_with_mode(col)
print(df[fill_mode_cols].head())
print(df[fill_mode_cols].isnull().sum())

In [ ]:
for col in delete_values_nah:
    df = df[(df["TARGET"] == 1) | ((df["TARGET"] == 0) & df[col].notna())]


In [ ]:
for col in delete_values_nah:
        print(f'MEAN của {col}: {df[col].mean()}')
        print(f'MEDIAN của {col}: {df[col].median()}')

In [ ]:
for col in delete_values_nah:
    df_nan = df[col][df[col].isnull() & (df["TARGET"] == 1)]
    print(df_nan.isnull().sum())
    print(df[col].isnull().sum())

In [ ]:
for col in delete_values_nah:
    if col == 'DAYS_LAST_PHONE_CHANGE':
        fillna_with_median(col)
    else:
        fillna_with_mean(col)


In [ ]:
df = df.drop(columns=delete_cols)

Gộp các giá trị cột 

In [ ]:
# CNT_CHILDREN: loại bỏ outlier >=6, gom >3 thành 1 nhóm
df = df[df["CNT_CHILDREN"] < 6]
df["CNT_CHILDREN_GROUP"] = df["CNT_CHILDREN"].apply(lambda x: x if x <= 3 else ">3")

In [ ]:
# NAME_EDUCATION_TYPE: gom thành 3 nhóm
edu_map = {
    "Lower secondary": "Basic_Education",
    "Secondary / secondary special": "Basic_Education",
    "Incomplete higher": "Incomplete higher",
    "Higher education": "Advanced_Education",
    "Academic degree": "Advanced_Education"
}
df["NAME_EDUCATION_GROUP"] = df["NAME_EDUCATION_TYPE"].map(edu_map)

In [ ]:
# ORGANIZATION_TYPE: gom nhóm theo danh sách
org_map = {
    "Trade: type 1":"Trade",
    "Trade: type 2":"Trade",
    "Trade: type 3":"Trade",
    "Trade: type 4":"Trade",
    "Trade: type 5":"Trade",
    "Trade: type 6":"Trade",
    "Trade: type 7":"Trade",
    "Industry: type 1":"Industry",
    "Industry: type 2":"Industry",
    "Industry: type 3":"Industry",
    "Industry: type 4":"Industry",
    "Industry: type 5":"Industry",
    "Industry: type 6":"Industry",
    "Industry: type 7":"Industry",
    "Industry: type 8":"Industry",
    "Industry: type 9":"Industry",
    "Industry: type 10":"Industry",
    "Industry: type 11":"Industry",
    "Industry: type 12":"Industry",
    "Industry: type 13":"Industry",
    "Transport: type 1":"Transport",
    "Transport: type 2":"Transport",
    "Transport: type 3":"Transport",
    "Transport: type 4":"Transport",
    "Business Entity Type 1":"Business Entity",
    "Business Entity Type 2":"Business Entity",
    "Business Entity Type 3":"Business Entity",
    "Self-employed":"Business Entity",
    "Government":"Public Sector",
    "Military":"Public Sector",
    "Police":"Public Sector",
    "Security Ministries":"Public Sector",
    "Emergency":"Public Sector",
    "Services":"Service/Professional",
    "Legal Services":"Service/Professional",
    "Advertising":"Service/Professional",
    "Cleaning":"Service/Professional",
    "Security":"Service/Professional",
    "Restaurant":"Service/Professional",
    "Hotel":"Service/Professional",
    "Postal":"Service/Professional",
    "Telecom":"Service/Professional",
    "Bank":"Service/Professional",
    "Insurance":"Service/Professional",
    "Mobile":"Service/Professional",
    "Realtor":"Service/Professional",
    "School":"Education/Culture",
    "University":"Education/Culture",
    "Kindergarten":"Education/Culture",
    "Culture":"Education/Culture",
    "Electricity":"Utility/Essential",
    "Medicine":"Utility/Essential",
    "Agriculture":"Utility/Essential",
    "Construction":"Utility/Essential",
    "Other":"Other/Unknown",
    "XNA":"Other/Unknown",
    "Housing":"Other/Unknown",""
    "Religion":"Other/Unknown"
}
df["ORGANIZATION_GROUP"] = df["ORGANIZATION_TYPE"].map(org_map)

In [ ]:
# NAME_INCOME_TYPE: gom thành 5 nhóm
income_map = {
    "Working":"Working",
    "Commercial associate":"Commercial associate",
    "Pensioner":"Pensioner",
    "State servant":"State servant",
    "Other":"Other"
}
df["NAME_INCOME_GROUP"] = df["NAME_INCOME_TYPE"].map(income_map)

In [ ]:
# NAME_HOUSING_TYPE: gom thành 3 nhóm
housing_map = {
    "House / apartment":"Owner",
    "With parents":"Dependent",
    "Rented apartment":"Renter/Other","Municipal apartment":"Renter/Other",
    "Office apartment":"Renter/Other","Co-op apartment":"Renter/Other"
}
df["NAME_HOUSING_GROUP"] = df["NAME_HOUSING_TYPE"].map(housing_map)

In [ ]:
# 6. CNT_FAM_MEMBERS: Sửa triệt để lỗi mixed types (ép float sạch về chuỗi số nguyên)
def fam_members_group(x):
    if pd.isna(x): return np.nan
    val = int(float(x))
    return str(val) if val <= 6 else ">6"
df["CNT_FAM_MEMBERS_GROUP"] = df["CNT_FAM_MEMBERS"].apply(fam_members_group)

In [ ]:
df["DEF_60_CNT_SOCIAL_CIRCLE"].value_counts()

In [ ]:
# DEF_30_CNT_SOCIAL_CIRCLE: gom thành 4 nhóm
def def30_group(x):
    if x == 0: return "0"
    elif x == 1: return "1"
    elif 2 <= x <= 4: return "2-4"
    else: return ">=5"
df["DEF_30_GROUP"] = df["DEF_30_CNT_SOCIAL_CIRCLE"].apply(def30_group)

def def60_group(x):
    if x == 0: return "0"
    elif x == 1: return "1"
    elif x == 2: return "2"
    else: return ">=3"
df["DEF_60_GROUP"] = df["DEF_60_CNT_SOCIAL_CIRCLE"].apply(def60_group)

def OBS30_group(x):
    if x == 0: return "0"
    elif x == 1: return "1"
    elif x == 2: return "2"
    else: return ">= 3"
df["OBS_30_GROUP"] = df["OBS_30_CNT_SOCIAL_CIRCLE"].apply(def30_group)

def OBS60_group(x):
    if x == 0: return "0"
    elif x == 1: return "1"
    elif x == 2: return "2"
    else: return ">= 3"
df["OBS_60_GROUP"] = df["OBS_60_CNT_SOCIAL_CIRCLE"].apply(def30_group)

# AMT_REQ_CREDIT_BUREAU_YEAR: gom thành 5 nhóm
def bureau_year_group(x):
    if x == 0: return "0"
    elif x == 1: return "1"
    elif 2 <= x <= 3: return "2-3"
    elif 4 <= x <= 6: return "4-6"
    else: return ">=7"
df["BUREAU_YEAR_GROUP"] = df["AMT_REQ_CREDIT_BUREAU_YEAR"].apply(bureau_year_group)

# YEARS_BEGINEXPLUATATION_AVG: gom thành nhóm 0,1,2,3,4,>=5
def years_group(x):
    if x <= 0: return "0"
    elif x == 1: return "1"
    elif x == 2: return "2"
    elif x == 3: return "3"
    elif x == 4: return "4"
    else: return ">=5"
df["YEARS_BEGIN_GROUP"] = df["YEARS_BEGINEXPLUATATION_AVG"].apply(years_group)

In [ ]:
df.to_csv("Clean_data_v2.csv",index = False)